# Transition dynamics and \(R_D\) analysis

Notebook per reconstruir les transicions stop-to-stop a partir de `saved_df/df_dynamics_with_oof.csv`, calcular les matrius de transició i estimar el ratio de pressió cap al discomfort \(R_D\) de dues maneres:

\[
R_D^{\mathrm{strict}} = \frac{C\to U}{U\to C}
\]

i

\[
R_D^{\mathrm{broad}} = \frac{(C,N)\to U}{U\to (C,N)}.
\]

El notebook també genera els plots paper-style que estàvem utilitzant: matrius per règim d'arribada, matrius per règim de sortida, same-regime persistence check, route geometry i \(R_D\) per bins d'igual amplada.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable


DATA_PATH = Path("../../data/df_dynamics_with_oof.csv")

OUT_DIR = Path("transition_outputs")
CSV_DIR = OUT_DIR / "csv"
FIG_DIR = OUT_DIR / "figures"

CSV_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Data path:", DATA_PATH)
print("Output folder:", OUT_DIR)

## 1. Global configuration

Thresholds retained in the TFM text:

- \(T_{\mathrm{corr}}\): 26.10 and 30.77 °C
- \(T_{\mathrm{env}}\): 24.62 and 30.83 effective °C

The code uses the three-state space:

\[
C, N, U = \mathrm{comfortable}, \mathrm{neutral}, \mathrm{uncomfortable}.
\]

In [ ]:
STATE_ORDER = ["comfortable", "neutral", "uncomfortable"]
STATE_LABELS = ["C", "N", "U"]

REGIME_LABELS = ["lower_T", "neutral_T", "higher_T"]
REGIME_TITLES = {
    "lower_T": "Cool",
    "neutral_T": "Central",
    "higher_T": "Hot",
}

COORDS = {
    "Tcorr": {
        "label": r"$T_{\mathrm{corr}}$",
        "start_col": "T_corr_start",
        "end_col": "T_corr_end",
        "mean_col": "T_corr_mean",
        "bins": [-np.inf, 26.10, 30.77, np.inf],
    },
    "Tenv": {
        "label": r"$T_{\mathrm{env}}$",
        "start_col": "T_env_start",
        "end_col": "T_env_end",
        "mean_col": "T_env_mean",
        "bins": [-np.inf, 24.62, 30.83, np.inf],
    },
}

K_MIX, K_SUN, K_WIND = 3.0, 9.5, 16.0

WIND_MAP = {
    "It's not windy": 0.0,
    "A very light wind": 0.25,
    "A light wind": 0.5,
    "A moderate wind": 0.75,
    "A strong wind": 1.0,
}


# ---------------------------------------------------------------------
# USER SETTINGS FOR R_D
# ---------------------------------------------------------------------
# Main coordinate for the R_D figure.
# Good options:
#   "T_env_end"    -> arrival effective environmental temperature
#   "T_env_mean"   -> mean temperature along the transition
#   "T_env_start"  -> departure effective environmental temperature
#   "T_corr_end"   -> arrival corrected temperature
RD_X_COL = "T_env_end"

# Number of bins to compute. You can add/remove values freely.
RD_BIN_NUMBERS = (3, 4, 5, 6, 8, 10, 12)

# Number of bins to show in the final plots.
# This does not need to match all computed bins.
RD_PLOT_BIN_NUMBERS = (3, 5, 8)

# "quantile" gives bins with approximately the same number of transitions.
# "equal_width" gives equally spaced temperature intervals.
RD_BINNING = "quantile"

# Bootstrap settings. Cluster bootstrap resamples whole participant-walk trajectories,
# preserving the within-walk transition structure more faithfully than row bootstrap.
RD_RUN_BOOTSTRAP = True
RD_BOOTSTRAP_N = 1000
RD_BOOTSTRAP_CLUSTER_COLS = ("ID", "walk_id")
RD_BOOTSTRAP_CI = (2.5, 97.5)
RD_RANDOM_SEED = 123

# Column used on the x-axis. "x_median" is usually better for quantile bins;
# "x_mid" is usually better for equal-width bins.
RD_X_PLOT_COL = "x_median" if RD_BINNING == "quantile" else "x_mid"


In [ ]:
def paper_style_default_font():
    """Paper-like style, but keeping the default DejaVu Sans font."""
    mpl.rcParams.update(mpl.rcParamsDefault)

    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "mathtext.fontset": "dejavusans",

        "font.size": 12,
        "axes.labelsize": 11,
        "axes.titlesize": 11,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 9,

        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,

        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
    })


paper_style_default_font()

BLUE_CMAP = LinearSegmentedColormap.from_list(
    "comfort_blues",
    ["#f7fbff", "#deebf7", "#9ecae1", "#3182bd", "#08519c", "#08306b"]
)

BLUE_LIGHT = "#6BAED6"
BLUE_DARK = "#08519C"
BLUE_MID = "#3182BD"

## 2. Load and prepare the data

This reproduces the construction you were using:

\[
T_{\mathrm{env}} =
T_{\mathrm{corr}}
+ k_{\mathrm{mix}} f_{\mathrm{mix}}
+ k_{\mathrm{sun}} f_{\mathrm{sun}}
- k_{\mathrm{wind}} \mathrm{wind}_{\mathrm{sc}} .
\]

In [ ]:
def load_and_prepare_data(path=DATA_PATH):
    df_prepared = pd.read_csv(path)

    required = [
        "ID", "stop_idx", "walk_id", "comfort3",
        "<T-T_fixed+<T>>", "sun", "wind"
    ]

    missing = [c for c in required if c not in df_prepared.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df1 = df_prepared.copy()
    df1 = df1.dropna(subset=required).copy()

    df1["T_corr"] = df1["<T-T_fixed+<T>>"]

    df1["f_mix"] = (df1["sun"] == "In a mixture of sun and shadow").astype(float)
    df1["f_sun"] = (df1["sun"] == "In full sun").astype(float)
    df1["wind_sc"] = df1["wind"].map(WIND_MAP)

    df1 = df1.dropna(subset=["wind_sc"]).copy()

    df1["T_env"] = (
        df1["T_corr"]
        + K_MIX * df1["f_mix"]
        + K_SUN * df1["f_sun"]
        - K_WIND * df1["wind_sc"]
    )

    return df1


df1 = load_and_prepare_data(DATA_PATH)

print("Prepared shape:", df1.shape)
df1.head()

## 3. Build stop-to-stop transitions

Transitions are constructed within \((\mathrm{ID}, \mathrm{walk\_id})\), never across walks. We keep only consecutive stops:

\[
\mathrm{step\_gap}=1.
\]

By default, the analysis also keeps the same restriction you were using:

\[
\mathrm{stop\_idx} \leq 5.
\]

In [ ]:
def build_transitions(df1, stop_idx_max=5):
    state_num_map = {s: i for i, s in enumerate(STATE_ORDER)}

    df = df1.copy()
    df = df.sort_values(["ID", "walk_id", "stop_idx"]).copy()

    df["state"] = df["comfort3"]
    df["state_num"] = df["state"].map(state_num_map)
    df = df.dropna(subset=["state_num"]).copy()

    df["T_corr_start"] = df["T_corr"]
    df["T_env_start"] = df["T_env"]

    group_cols = ["ID", "walk_id"]

    df["T_corr_end"] = df.groupby(group_cols)["T_corr"].shift(-1)
    df["T_env_end"] = df.groupby(group_cols)["T_env"].shift(-1)

    df["T_corr_mean"] = 0.5 * (df["T_corr_start"] + df["T_corr_end"])
    df["T_env_mean"] = 0.5 * (df["T_env_start"] + df["T_env_end"])

    df["T_corr_delta"] = df["T_corr_end"] - df["T_corr_start"]
    df["T_env_delta"] = df["T_env_end"] - df["T_env_start"]

    df["next_state"] = df.groupby(group_cols)["state"].shift(-1)
    df["next_state_num"] = df.groupby(group_cols)["state_num"].shift(-1)
    df["next_stop_idx"] = df.groupby(group_cols)["stop_idx"].shift(-1)

    df["step_gap"] = df["next_stop_idx"] - df["stop_idx"]

    transitions = df[df["step_gap"] == 1].copy()

    if stop_idx_max is not None:
        transitions = transitions[transitions["stop_idx"] <= stop_idx_max].copy()

    transitions["delta"] = transitions["next_state_num"] - transitions["state_num"]

    transitions = transitions.dropna(
        subset=[
            "state", "next_state",
            "T_corr_start", "T_corr_end",
            "T_env_start", "T_env_end"
        ]
    ).copy()

    return transitions


transitions = build_transitions(df1, stop_idx_max=5)

print("Transitions:", transitions.shape)
display(transitions[["state", "next_state", "T_corr_start", "T_corr_end", "T_env_start", "T_env_end", "delta"]].head())

transitions.to_csv(CSV_DIR / "transitions_built.csv", index=False)

## 4. Transition matrix helpers

In [ ]:
def add_regime_bins(transitions):
    """Add start, end and mean regime bins for Tcorr and Tenv."""
    df = transitions.copy()

    for coord, cfg in COORDS.items():
        df[f"{coord}_start_bin"] = pd.cut(
            df[cfg["start_col"]],
            bins=cfg["bins"],
            labels=REGIME_LABELS,
            right=False,
        )

        df[f"{coord}_end_bin"] = pd.cut(
            df[cfg["end_col"]],
            bins=cfg["bins"],
            labels=REGIME_LABELS,
            right=False,
        )

        df[f"{coord}_mean_bin"] = pd.cut(
            df[cfg["mean_col"]],
            bins=cfg["bins"],
            labels=REGIME_LABELS,
            right=False,
        )

    return df


def compute_transition_matrix(sub, state_order=STATE_ORDER):
    """Count and row-normalised transition matrix."""
    cm = pd.crosstab(
        sub["state"],
        sub["next_state"]
    ).reindex(index=state_order, columns=state_order, fill_value=0)

    cm_prob = cm.div(cm.sum(axis=1), axis=0)

    return cm, cm_prob


def save_transition_matrices_by_bin(
    transitions,
    coord="Tcorr",
    bin_kind="end",
    prefix=None,
):
    """
    Save count/probability matrices grouped by a given thermal regime bin.

    bin_kind: "start", "end", or "mean".
    """
    df = add_regime_bins(transitions)
    bin_col = f"{coord}_{bin_kind}_bin"

    if prefix is None:
        prefix = f"{coord}_{bin_kind}"

    out = {}

    for b, sub in df.groupby(bin_col, observed=True):
        cm, cm_prob = compute_transition_matrix(sub)
        out[str(b)] = {"counts": cm, "prob": cm_prob}

        cm.to_csv(CSV_DIR / f"cm_counts_{prefix}_{b}.csv")
        cm_prob.to_csv(CSV_DIR / f"cm_prob_{prefix}_{b}.csv")

    return out


tmats_Tcorr_end = save_transition_matrices_by_bin(transitions, coord="Tcorr", bin_kind="end")
tmats_Tenv_end = save_transition_matrices_by_bin(transitions, coord="Tenv", bin_kind="end")

## 5. Plot transition matrices

Main-text figure: matrices grouped by arrival regime \(T_{i+1}\).

Appendix figure: matrices grouped by departure regime \(T_i\).

In [ ]:
def plot_transition_matrices_by_regime(
    transitions,
    bin_kind="end",
    output_pdf=None,
    output_png=None,
    vmin=0,
    vmax=0.85,
):
    """
    2 x 3 panel of transition matrices.

    Rows:
        Tcorr, Tenv

    Columns:
        lower_T, neutral_T, higher_T

    bin_kind:
        "end"   -> arrival temperature T_{i+1}
        "start" -> departure temperature T_i
        "mean"  -> mean transition temperature
    """
    if bin_kind not in {"start", "end", "mean"}:
        raise ValueError("bin_kind must be 'start', 'end', or 'mean'.")

    paper_style_default_font()

    df = add_regime_bins(transitions)

    coord_info = [
        {
            "name": "Tcorr",
            "label": {
                "start": r"$T_{\mathrm{corr},i}$",
                "end": r"$T_{\mathrm{corr},i+1}$",
                "mean": r"$\overline{T}_{\mathrm{corr}}$",
            }[bin_kind],
            "bin_col": f"Tcorr_{bin_kind}_bin",
        },
        {
            "name": "Tenv",
            "label": {
                "start": r"$T_{\mathrm{env},i}$",
                "end": r"$T_{\mathrm{env},i+1}$",
                "mean": r"$\overline{T}_{\mathrm{env}}$",
            }[bin_kind],
            "bin_col": f"Tenv_{bin_kind}_bin",
        },
    ]

    fig, axes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(7.2, 4.6),
        sharex=True,
        sharey=True,
    )

    norm = Normalize(vmin=vmin, vmax=vmax)

    for row, coord in enumerate(coord_info):
        for col, regime in enumerate(REGIME_LABELS):
            ax = axes[row, col]

            sub = df[df[coord["bin_col"]] == regime].copy()
            cm, cm_prob = compute_transition_matrix(sub)
            mat = cm_prob.values.astype(float)

            ax.imshow(
                mat,
                aspect="equal",
                cmap=BLUE_CMAP,
                norm=norm,
                interpolation="nearest",
            )

            ax.set_xticks(range(len(STATE_LABELS)))
            ax.set_yticks(range(len(STATE_LABELS)))
            ax.set_xticklabels(STATE_LABELS)
            ax.set_yticklabels(STATE_LABELS)
            ax.tick_params(axis="both", length=0)

            if row == 0:
                ax.set_title(REGIME_TITLES[regime], pad=8)

            if col == 0:
                ax.set_ylabel(coord["label"] + "\n" + r"$s_i$")

            if row == 1:
                ax.set_xlabel(r"$s_{i+1}$")

            ax.text(
                0.03,
                0.97,
                rf"$n={len(sub)}$",
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=8,
                bbox=dict(
                    boxstyle="round,pad=0.15",
                    facecolor="white",
                    edgecolor="none",
                    alpha=0.85,
                ),
            )

            for i in range(mat.shape[0]):
                for j in range(mat.shape[1]):
                    p = mat[i, j]
                    k = cm.values[i, j]

                    txt = "" if np.isnan(p) else f"{p:.2f}\n({k})"
                    text_color = "white" if (not np.isnan(p) and p >= 0.50) else "black"

                    ax.text(
                        j, i,
                        txt,
                        ha="center",
                        va="center",
                        fontsize=8.5,
                        color=text_color,
                    )

    cbar_ax = fig.add_axes([0.91, 0.18, 0.018, 0.66])
    sm = ScalarMappable(norm=norm, cmap=BLUE_CMAP)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label(r"$P(s_{i+1}\mid s_i)$", rotation=90, labelpad=10)

    fig.subplots_adjust(
        left=0.08,
        right=0.88,
        bottom=0.11,
        top=0.90,
        wspace=0.18,
        hspace=0.25,
    )

    if output_pdf is not None:
        fig.savefig(output_pdf)
    if output_png is not None:
        fig.savefig(output_png, dpi=300)

    plt.show()
    return fig


fig_arrival = plot_transition_matrices_by_regime(
    transitions,
    bin_kind="end",
    output_pdf=FIG_DIR / "transition_matrices_arrival_Tcorr_Tenv.pdf",
    output_png=FIG_DIR / "transition_matrices_arrival_Tcorr_Tenv.png",
)

fig_departure = plot_transition_matrices_by_regime(
    transitions,
    bin_kind="start",
    output_pdf=FIG_DIR / "transition_matrices_departure_Tcorr_Tenv.pdf",
    output_png=FIG_DIR / "transition_matrices_departure_Tcorr_Tenv.png",
)

## 6. Same-regime persistence check

This keeps only transitions for which start and end are in the same thermal regime:

\[
T_i \in r,\quad T_{i+1}\in r.
\]

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan

    p = k / n
    denom = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    half = z * np.sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2)))

    lo = (centre - half) / denom
    hi = (centre + half) / denom

    return lo, hi


def channel_prob_with_ci(sub, from_state, to_state):
    risk = sub[sub["state"] == from_state]
    n = len(risk)

    if n == 0:
        return np.nan, np.nan, np.nan, 0, 0

    k = (risk["next_state"] == to_state).sum()
    p = k / n
    lo, hi = wilson_ci(k, n)

    return p, lo, hi, k, n


def compute_same_regime_channels(transitions):
    df = add_regime_bins(transitions)

    channels = [
        ("comfortable", "comfortable", "C_to_C", r"$C\to C$"),
        ("uncomfortable", "uncomfortable", "U_to_U", r"$U\to U$"),
        ("comfortable", "uncomfortable", "C_to_U", r"$C\to U$"),
        ("uncomfortable", "comfortable", "U_to_C", r"$U\to C$"),
    ]

    rows = []

    for coord, cfg in COORDS.items():
        start_bin = f"{coord}_start_bin"
        end_bin = f"{coord}_end_bin"

        for regime in REGIME_LABELS:
            sub = df[
                (df[start_bin] == regime) &
                (df[end_bin] == regime)
            ].copy()

            for from_state, to_state, channel, channel_label in channels:
                p, lo, hi, k, n = channel_prob_with_ci(sub, from_state, to_state)

                rows.append({
                    "coordinate": coord,
                    "coordinate_label": cfg["label"],
                    "regime": regime,
                    "regime_title": REGIME_TITLES[regime],
                    "channel": channel,
                    "channel_label": channel_label,
                    "from": from_state,
                    "to": to_state,
                    "k": k,
                    "n": n,
                    "p": p,
                    "ci_low": lo,
                    "ci_high": hi,
                    "n_transitions": len(sub),
                })

    return pd.DataFrame(rows)


same_regime_df = compute_same_regime_channels(transitions)
same_regime_df.to_csv(CSV_DIR / "same_regime_channels.csv", index=False)

display(
    same_regime_df[
        same_regime_df["channel"].isin(["C_to_C", "U_to_U"])
    ][
        ["coordinate", "regime", "channel", "k", "n", "p", "ci_low", "ci_high", "n_transitions"]
    ]
)

In [ ]:
def plot_same_regime_persistence(
    same_regime_df,
    output_pdf=None,
    output_png=None,
):
    paper_style_default_font()

    plot_df = same_regime_df[
        same_regime_df["channel"].isin(["C_to_C", "U_to_U"])
    ].copy()

    coords = ["Tcorr", "Tenv"]
    coord_titles = {
        "Tcorr": r"$T_{\mathrm{corr}}$",
        "Tenv": r"$T_{\mathrm{env}}$",
    }

    x = np.arange(len(REGIME_LABELS))

    offsets = {"C_to_C": -0.07, "U_to_U": +0.07}
    colors = {"C_to_C": BLUE_LIGHT, "U_to_U": BLUE_DARK}
    markers = {"C_to_C": "o", "U_to_U": "s"}
    labels = {"C_to_C": r"$C\to C$", "U_to_U": r"$U\to U$"}

    fig, axes = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(7.0, 3.15),
        sharey=True,
    )

    for ax, coord in zip(axes, coords):
        sub_coord = plot_df[plot_df["coordinate"] == coord].copy()

        for channel in ["C_to_C", "U_to_U"]:
            sub_ch = (
                sub_coord[sub_coord["channel"] == channel]
                .set_index("regime")
                .reindex(REGIME_LABELS)
                .reset_index()
            )

            y = sub_ch["p"].values
            yerr_low = y - sub_ch["ci_low"].values
            yerr_high = sub_ch["ci_high"].values - y
            yerr = np.vstack([yerr_low, yerr_high])

            xpos = x + offsets[channel]

            ax.errorbar(
                xpos,
                y,
                yerr=yerr,
                fmt=markers[channel],
                color=colors[channel],
                markerfacecolor=colors[channel],
                markeredgecolor="black",
                markeredgewidth=0.4,
                elinewidth=1.0,
                capsize=3,
                markersize=5,
                linewidth=1.0,
                label=labels[channel],
                zorder=3,
            )

            ax.plot(
                xpos,
                y,
                color=colors[channel],
                linewidth=1.0,
                alpha=0.95,
                zorder=2,
            )

      

        ax.set_title(coord_titles[coord], pad=8, fontsize = 15)
        ax.set_xticks(x)
        ax.set_xticklabels([REGIME_TITLES[r] for r in REGIME_LABELS], fontsize = 13)
        ax.set_ylim(0, 0.9)
        ax.set_xlabel("")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        

        

    axes[0].set_ylabel("Persistence probability", fontsize= 15)

    handles, labels_ = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels_,
        loc="upper center",
        ncol=2,
        frameon=False,
        bbox_to_anchor=(0.5, 1.02),
        fontsize = 13
    )

    fig.supxlabel("Thermal regime", y=0.035, fontsize=14)

    fig.subplots_adjust(
        left=0.085,
        right=0.985,
        bottom=0.265,
        top=0.80,
        wspace=0.13,
    )

    if output_pdf is not None:
        fig.savefig(output_pdf)
    if output_png is not None:
        fig.savefig(output_png, dpi=300)

    plt.show()
    return fig


fig_same_regime = plot_same_regime_persistence(
    same_regime_df,
    output_pdf=FIG_DIR / "same_regime_persistence_check.pdf",
    output_png=FIG_DIR / "same_regime_persistence_check.png",
)

## 7. Route geometry: where transitions come from and where they end

These matrices do not show comfort transitions. They show:

\[
P(T_{i+1}\ \mathrm{bin}\mid T_i\ \mathrm{bin}).
\]

This is useful for documenting that conditioning on \(T_i\) and \(T_{i+1}\) selects different subsets of transitions.

In [ ]:
def compute_route_geometry_matrix(
    transitions,
    start_bin_col,
    end_bin_col,
    regime_labels=REGIME_LABELS,
):
    counts = pd.crosstab(
        transitions[start_bin_col],
        transitions[end_bin_col]
    ).reindex(index=regime_labels, columns=regime_labels, fill_value=0)

    probs = counts.div(counts.sum(axis=1), axis=0)

    return counts, probs


def plot_route_geometry(
    transitions,
    onset_risk_only=False,
    output_pdf=None,
    output_png=None,
    vmin=0,
    vmax=1,
):
    paper_style_default_font()

    df = add_regime_bins(transitions)

    if onset_risk_only:
        df = df[df["state"] == "comfortable"].copy()
        suffix = "onset_risk"
    else:
        suffix = "all_transitions"

    coord_info = [
        {
            "name": "Tcorr",
            "title": r"$T_{\mathrm{corr}}$",
            "start_bin": "Tcorr_start_bin",
            "end_bin": "Tcorr_end_bin",
        },
        {
            "name": "Tenv",
            "title": r"$T_{\mathrm{env}}$",
            "start_bin": "Tenv_start_bin",
            "end_bin": "Tenv_end_bin",
        },
    ]

    fig, axes = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(6.8, 3.2),
        sharex=True,
        sharey=True,
    )

    norm = Normalize(vmin=vmin, vmax=vmax)

    all_out = {}

    for ax, coord in zip(axes, coord_info):
        counts, probs = compute_route_geometry_matrix(
            df,
            start_bin_col=coord["start_bin"],
            end_bin_col=coord["end_bin"],
        )

        all_out[coord["name"]] = {"counts": counts, "prob": probs}

        counts.to_csv(CSV_DIR / f"route_geometry_counts_{coord['name']}_{suffix}.csv")
        probs.to_csv(CSV_DIR / f"route_geometry_prob_{coord['name']}_{suffix}.csv")

        mat = probs.values.astype(float)

        ax.imshow(
            mat,
            aspect="equal",
            cmap=BLUE_CMAP,
            norm=norm,
            interpolation="nearest",
        )

        ax.set_title(coord["title"], pad=8)

        ax.set_xticks(range(len(REGIME_LABELS)))
        ax.set_yticks(range(len(REGIME_LABELS)))

        ax.set_xticklabels(
            [REGIME_TITLES[r] for r in REGIME_LABELS],
            rotation=25,
            ha="right",
        )
        ax.set_yticklabels([REGIME_TITLES[r] for r in REGIME_LABELS])

        ax.set_xlabel(r"$T_{i+1}$ regime")
        ax.tick_params(axis="both", length=0)

        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                p = mat[i, j]
                k = counts.values[i, j]

                txt = "" if np.isnan(p) else f"{p:.2f}\n({k})"
                text_color = "white" if (not np.isnan(p) and p >= 0.50) else "black"

                ax.text(
                    j, i,
                    txt,
                    ha="center",
                    va="center",
                    fontsize=8.5,
                    color=text_color,
                )

        ax.text(
            0.03,
            0.97,
            rf"$n={len(df)}$",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8,
            bbox=dict(
                boxstyle="round,pad=0.15",
                facecolor="white",
                edgecolor="none",
                alpha=0.85,
            ),
        )

    axes[0].set_ylabel(r"$T_i$ regime")

    cbar_ax = fig.add_axes([0.92, 0.20, 0.018, 0.62])
    sm = ScalarMappable(norm=norm, cmap=BLUE_CMAP)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label(
        r"$P(T_{i+1}\ \mathrm{bin}\mid T_i\ \mathrm{bin})$",
        rotation=90,
        labelpad=10,
    )

    fig.subplots_adjust(
        left=0.10,
        right=0.89,
        bottom=0.20,
        top=0.84,
        wspace=0.18,
    )

    if output_pdf is not None:
        fig.savefig(output_pdf)
    if output_png is not None:
        fig.savefig(output_png, dpi=300)

    plt.show()
    return fig, all_out


fig_route_all, route_all = plot_route_geometry(
    transitions,
    onset_risk_only=False,
    output_pdf=FIG_DIR / "route_geometry_all_transitions.pdf",
    output_png=FIG_DIR / "route_geometry_all_transitions.png",
)

fig_route_onset, route_onset = plot_route_geometry(
    transitions,
    onset_risk_only=True,
    output_pdf=FIG_DIR / "route_geometry_onset_risk.pdf",
    output_png=FIG_DIR / "route_geometry_onset_risk.png",
)

## 8. \(R_D\) in configurable bins with bootstrap uncertainty

This section computes \(R_D\) for a selected coordinate, by default \(T_{\mathrm{env},i+1}\).

Two definitions are retained:

\[
R_D^{\mathrm{strict}} = \frac{C\to U}{U\to C}
\]

and

\[
R_D^{\mathrm{broad}} = \frac{(C,N)\to U}{U\to (C,N)}.
\]

The binning is now configurable from the global settings above:

- `RD_X_COL`: coordinate used for \(R_D\), e.g. `"T_env_end"`;
- `RD_BIN_NUMBERS`: bin counts to compute;
- `RD_PLOT_BIN_NUMBERS`: bin counts to display;
- `RD_BINNING="quantile"`: bins contain approximately the same number of transitions;
- `RD_BINNING="equal_width"`: bins have equal temperature width.

Bootstrap errors are computed by resampling whole `(ID, walk_id)` trajectories with replacement. This gives a walk/participant-level uncertainty estimate rather than treating every transition as fully independent.


In [ ]:
RD_X_COL = "T_corr_end"

def safe_div(a, b):
    if b == 0 or pd.isna(b):
        return np.nan
    return a / b


def compute_RD_from_transition_matrix(cm, cm_prob):
    """
    Computes two R_D definitions from a transition count matrix
    and its row-normalised probability matrix.
    """
    C = "comfortable"
    N = "neutral"
    U = "uncomfortable"

    k_CU = cm.loc[C, U]
    k_UC = cm.loc[U, C]

    k_CNU = cm.loc[C, U] + cm.loc[N, U]
    k_UCN = cm.loc[U, C] + cm.loc[U, N]

    RD_strict_counts = safe_div(k_CU, k_UC)
    RD_broad_counts = safe_div(k_CNU, k_UCN)

    p_CU = cm_prob.loc[C, U]
    p_UC = cm_prob.loc[U, C]

    n_C = cm.loc[C].sum()
    n_N = cm.loc[N].sum()
    n_U = cm.loc[U].sum()

    p_CNU = safe_div(k_CNU, n_C + n_N)
    p_UCN = safe_div(k_UCN, n_U)

    RD_strict_prob = safe_div(p_CU, p_UC)
    RD_broad_prob = safe_div(p_CNU, p_UCN)

    return {
        "k_CU": k_CU,
        "k_UC": k_UC,
        "k_CNU": k_CNU,
        "k_UCN": k_UCN,

        "n_C_risk": n_C,
        "n_N_risk": n_N,
        "n_U_risk": n_U,
        "n_CN_risk": n_C + n_N,

        "p_CU": p_CU,
        "p_UC": p_UC,
        "p_CNU": p_CNU,
        "p_UCN": p_UCN,

        "RD_strict_counts": RD_strict_counts,
        "RD_broad_counts": RD_broad_counts,
        "RD_strict_prob": RD_strict_prob,
        "RD_broad_prob": RD_broad_prob,
    }


RD_METRICS = [
    "RD_strict_counts",
    "RD_broad_counts",
    "RD_strict_prob",
    "RD_broad_prob",
]


def make_bin_edges(values, n_bins, method="quantile"):
    """
    Build fixed bin edges from the original data.

    method="quantile" produces approximately equal-count bins.
    method="equal_width" produces equally spaced bins.
    """
    x = pd.Series(values).dropna().astype(float)

    if x.empty:
        raise ValueError("Cannot build bins: x_col has no finite values.")

    if n_bins < 1:
        raise ValueError("n_bins must be >= 1.")

    if method == "quantile":
        edges = np.quantile(x, np.linspace(0, 1, n_bins + 1))
        edges = np.unique(edges)

        if len(edges) < 2:
            raise ValueError(
                "Cannot build quantile bins because all values are identical."
            )

    elif method == "equal_width":
        xmin, xmax = x.min(), x.max()

        if np.isclose(xmin, xmax):
            raise ValueError(
                "Cannot build equal-width bins because all values are identical."
            )

        edges = np.linspace(xmin, xmax, n_bins + 1)

    else:
        raise ValueError("method must be 'quantile' or 'equal_width'.")

    # Expand extremes very slightly so the min/max values are always included.
    eps = np.finfo(float).eps * max(1.0, float(np.nanmax(np.abs(edges))))
    edges = edges.astype(float)
    edges[0] -= eps
    edges[-1] += eps

    return edges


def add_fixed_bins(df, x_col, edges, bin_col="_rd_bin_id"):
    """Assign observations to fixed numeric bins."""
    out = df.copy()
    out[bin_col] = pd.cut(
        out[x_col].astype(float),
        bins=edges,
        labels=False,
        include_lowest=True,
        duplicates="drop",
    )
    out = out.dropna(subset=[bin_col]).copy()
    out[bin_col] = out[bin_col].astype(int)
    return out


def _interval_label(left, right):
    return f"({left:.3g}, {right:.3g}]"


def _compute_RD_rows_from_binned(
    binned,
    x_col,
    n_bins_requested,
    edges,
    binning,
    state_order=STATE_ORDER,
    bin_col="_rd_bin_id",
):
    """Compute base R_D rows and transition matrices for already-binned data."""
    rows = []
    tmats = {}
    actual_n_bins = len(edges) - 1

    for bin_id in range(actual_n_bins):
        sub = binned[binned[bin_col] == bin_id].copy()

        cm, cm_prob = compute_transition_matrix(sub, state_order=state_order)
        rd_vals = compute_RD_from_transition_matrix(cm, cm_prob)
        tmats[bin_id] = {"counts": cm, "prob": cm_prob}

        left = edges[bin_id]
        right = edges[bin_id + 1]

        if len(sub) > 0:
            x_min_obs = sub[x_col].min()
            x_max_obs = sub[x_col].max()
            x_median = sub[x_col].median()
        else:
            x_min_obs = np.nan
            x_max_obs = np.nan
            x_median = 0.5 * (left + right)

        row = {
            "x_col": x_col,
            "binning": binning,
            "n_bins": n_bins_requested,
            "n_bins_actual": actual_n_bins,
            "bin_id": bin_id,
            "T_bin": _interval_label(left, right),
            "x_edge_left": left,
            "x_edge_right": right,
            "x_min": x_min_obs,
            "x_max": x_max_obs,
            "x_mid": 0.5 * (left + right),
            "x_median": x_median,
            "n_transitions": len(sub),
        }

        row.update(rd_vals)
        rows.append(row)

    return pd.DataFrame(rows), tmats


def _bootstrap_RD_for_binned(
    binned,
    n_bins_requested,
    actual_n_bins,
    n_boot=1000,
    cluster_cols=("ID", "walk_id"),
    seed=123,
    ci=(2.5, 97.5),
    state_order=STATE_ORDER,
    bin_col="_rd_bin_id",
):
    """
    Cluster bootstrap for R_D.

    The bin edges are fixed before resampling. Each bootstrap replicate resamples
    complete clusters, where a cluster is usually one participant in one walk.
    """
    if n_boot is None or n_boot <= 0:
        return pd.DataFrame({
            "n_bins": [n_bins_requested] * actual_n_bins,
            "bin_id": list(range(actual_n_bins)),
        })

    available_cluster_cols = [c for c in cluster_cols if c in binned.columns]

    if len(available_cluster_cols) == 0:
        # Fallback: row bootstrap if cluster columns are missing.
        groups = [binned.iloc[[i]].copy() for i in range(len(binned))]
    else:
        groups = [
            g.copy()
            for _, g in binned.groupby(available_cluster_cols, sort=False, dropna=False)
        ]

    if len(groups) == 0:
        return pd.DataFrame({
            "n_bins": [n_bins_requested] * actual_n_bins,
            "bin_id": list(range(actual_n_bins)),
        })

    rng = np.random.default_rng(seed)
    boot_rows = []

    for boot_id in range(n_boot):
        sampled_idx = rng.integers(0, len(groups), size=len(groups))
        boot = pd.concat([groups[i] for i in sampled_idx], ignore_index=True)

        for bin_id in range(actual_n_bins):
            sub = boot[boot[bin_col] == bin_id].copy()
            cm, cm_prob = compute_transition_matrix(sub, state_order=state_order)
            rd_vals = compute_RD_from_transition_matrix(cm, cm_prob)

            row = {
                "boot_id": boot_id,
                "n_bins": n_bins_requested,
                "bin_id": bin_id,
            }
            for metric in RD_METRICS:
                row[metric] = rd_vals[metric]

            boot_rows.append(row)

    boot_df = pd.DataFrame(boot_rows)
    summary_rows = []

    q_low, q_high = ci

    for bin_id in range(actual_n_bins):
        row = {
            "n_bins": n_bins_requested,
            "bin_id": bin_id,
        }

        sub_bin = boot_df[boot_df["bin_id"] == bin_id]

        for metric in RD_METRICS:
            values = sub_bin[metric].replace([np.inf, -np.inf], np.nan).dropna()

            row[f"{metric}_boot_valid"] = len(values)

            if len(values) == 0:
                row[f"{metric}_low"] = np.nan
                row[f"{metric}_high"] = np.nan
                row[f"{metric}_boot_median"] = np.nan
            else:
                row[f"{metric}_low"] = np.nanpercentile(values, q_low)
                row[f"{metric}_high"] = np.nanpercentile(values, q_high)
                row[f"{metric}_boot_median"] = np.nanmedian(values)

        summary_rows.append(row)

    return pd.DataFrame(summary_rows)


def compute_RD_bins_like_transition_matrices(
    transitions,
    x_col,
    bin_numbers=(12, 10, 8, 6, 5),
    binning="quantile",
    bootstrap=True,
    n_boot=1000,
    cluster_cols=("ID", "walk_id"),
    ci=(2.5, 97.5),
    seed=123,
    state_order=STATE_ORDER,
    output_csv=None,
):
    """
    Compute R_D in configurable bins.

    Important choices:
    - binning="quantile": approximately the same number of transitions per bin.
    - binning="equal_width": equal-width intervals along x_col.
    - bootstrap=True: adds percentile confidence bands by cluster bootstrap.
    """
    if x_col not in transitions.columns:
        raise ValueError(
            f"x_col='{x_col}' is not in transitions. "
            f"Available temperature columns include: "
            f"{[c for c in transitions.columns if c.startswith('T_')]}"
        )

    all_rows = []
    all_tmats = {}

    base = transitions.dropna(subset=[x_col, "state", "next_state"]).copy()

    for n_bins in bin_numbers:
        edges = make_bin_edges(base[x_col], n_bins=n_bins, method=binning)
        actual_n_bins = len(edges) - 1

        if actual_n_bins != n_bins:
            print(
                f"Requested {n_bins} bins for {x_col}, but only "
                f"{actual_n_bins} unique bins could be built because of tied values."
            )

        binned = add_fixed_bins(base, x_col=x_col, edges=edges)
        rd_base, tmats = _compute_RD_rows_from_binned(
            binned=binned,
            x_col=x_col,
            n_bins_requested=n_bins,
            edges=edges,
            binning=binning,
            state_order=state_order,
        )

        if bootstrap:
            rd_ci = _bootstrap_RD_for_binned(
                binned=binned,
                n_bins_requested=n_bins,
                actual_n_bins=actual_n_bins,
                n_boot=n_boot,
                cluster_cols=cluster_cols,
                seed=seed + int(n_bins),
                ci=ci,
                state_order=state_order,
            )
            rd_base = rd_base.merge(rd_ci, on=["n_bins", "bin_id"], how="left")

        all_rows.append(rd_base)
        all_tmats[n_bins] = tmats

    out = pd.concat(all_rows, ignore_index=True)

    if output_csv is not None:
        out.to_csv(output_csv, index=False)

    return out, all_tmats


def xlabel_from_xcol(x_col):
    label_map = {
        "T_corr_start": r"$T_{\mathrm{corr},i}$",
        "T_corr_end": r"$T_{\mathrm{corr},i+1}$",
        "T_corr_mean": r"$\overline{T}_{\mathrm{corr}}$",
        "T_env_start": r"$T_{\mathrm{env},i}$",
        "T_env_end": r"$T_{\mathrm{env},i+1}$",
        "T_env_mean": r"$\overline{T}_{\mathrm{env}}$",
    }
    return label_map.get(x_col, x_col)


RD_OUTPUT_TAG = f"{RD_X_COL}_{RD_BINNING}"

RD_selected_bins, tmats_RD_selected_bins = compute_RD_bins_like_transition_matrices(
    transitions,
    x_col=RD_X_COL,
    bin_numbers=RD_BIN_NUMBERS,
    binning=RD_BINNING,
    bootstrap=RD_RUN_BOOTSTRAP,
    n_boot=RD_BOOTSTRAP_N,
    cluster_cols=RD_BOOTSTRAP_CLUSTER_COLS,
    ci=RD_BOOTSTRAP_CI,
    seed=RD_RANDOM_SEED,
    output_csv=CSV_DIR / f"RD_bins_{RD_OUTPUT_TAG}.csv",
)

print("R_D coordinate:", RD_X_COL)
print("Binning:", RD_BINNING)
print("Computed bin numbers:", RD_BIN_NUMBERS)
print("Plotted bin numbers:", RD_PLOT_BIN_NUMBERS)
print("Bootstrap:", RD_RUN_BOOTSTRAP, "| n_boot:", RD_BOOTSTRAP_N)

display_cols = [
    "n_bins", "bin_id", "T_bin", "x_median", "n_transitions",
    "n_C_risk", "n_U_risk",
    "RD_strict_prob", "RD_strict_prob_low", "RD_strict_prob_high",
    "RD_broad_prob", "RD_broad_prob_low", "RD_broad_prob_high",
]

available_display_cols = [c for c in display_cols if c in RD_selected_bins.columns]
display(RD_selected_bins[available_display_cols].head(30))

## 9. Plot \(R_D\) by selected bin numbers

The plot functions below read `RD_PLOT_BIN_NUMBERS`. Change that tuple in the configuration cell to decide which bin counts are shown, without recomputing all of the notebook logic. The bootstrap interval is drawn as a shaded band around each curve when the `_low` and `_high` columns exist.


In [ ]:
RD_X_COL = "T_corr_end"

def _select_plot_bins(rd_df, plot_bin_numbers=None):
    df = rd_df.copy()

    if plot_bin_numbers is not None:
        plot_bin_numbers = tuple(plot_bin_numbers)
        df = df[df["n_bins"].isin(plot_bin_numbers)].copy()

    if df.empty:
        raise ValueError(
            "No rows left to plot. Check RD_PLOT_BIN_NUMBERS against "
            "the n_bins values present in rd_df."
        )

    return df


def _dynamic_style_for_bins(nbin_values):
    nbin_values = list(nbin_values)
    cmap_values = np.linspace(0.35, 0.90, max(len(nbin_values), 2))
    colors = {
        nb: plt.cm.Blues(cmap_values[i])
        for i, nb in enumerate(nbin_values)
    }

    marker_cycle = ["o", "s", "^", "D", "v", "P", "X", "*", "<", ">"]
    markers = {
        nb: marker_cycle[i % len(marker_cycle)]
        for i, nb in enumerate(nbin_values)
    }

    return colors, markers


def _add_bootstrap_shadow(
    ax,
    sub,
    xcol,
    ycol,
    color,
    alpha=0.18,
):
    low_col = f"{ycol}_low"
    high_col = f"{ycol}_high"

    if low_col not in sub.columns or high_col not in sub.columns:
        return

    x = sub[xcol].astype(float).to_numpy()
    low = sub[low_col].astype(float).to_numpy()
    high = sub[high_col].astype(float).to_numpy()

    mask = np.isfinite(x) & np.isfinite(low) & np.isfinite(high)

    if mask.sum() >= 2:
        ax.fill_between(
            x[mask],
            low[mask],
            high[mask],
            color=color,
            alpha=alpha,
            linewidth=0,
            zorder=2,
        )
    elif mask.sum() == 1:
        # With one bin there is no area to fill, so draw a vertical interval.
        ax.vlines(
            x[mask],
            low[mask],
            high[mask],
            color=color,
            alpha=0.45,
            linewidth=1.0,
            zorder=2,
        )


def plot_single_RD_column_by_nbins(
    rd_df,
    ycol="RD_broad_prob",
    ylabel=r"$R_D$",
    xlabel=None,
    x_plot_col=None,
    plot_bin_numbers=None,
    output_pdf=None,
    output_png=None,
    ylim=None,
    shade_ci=True,
):
    paper_style_default_font()

    if xlabel is None:
        xlabel = xlabel_from_xcol(rd_df["x_col"].iloc[0])

    if x_plot_col is None:
        x_plot_col = RD_X_PLOT_COL if "RD_X_PLOT_COL" in globals() else "x_median"

    df = _select_plot_bins(rd_df, plot_bin_numbers)
    df = df.copy().sort_values(["n_bins", x_plot_col])
    nbin_values = sorted(df["n_bins"].unique())

    color_map, marker_map = _dynamic_style_for_bins(nbin_values)

    fig, ax = plt.subplots(figsize=(3.65, 2.75))

    for nb in nbin_values:
        sub = df[df["n_bins"] == nb].copy().sort_values(x_plot_col)
        color = color_map[nb]

        if shade_ci:
            _add_bootstrap_shadow(ax, sub, x_plot_col, ycol, color=color)

        ax.plot(
            sub[x_plot_col],
            sub[ycol],
            marker=marker_map[nb],
            markersize=4.2,
            markerfacecolor=color,
            markeredgecolor="black",
            markeredgewidth=0.35,
            color=color,
            linewidth=1.05,
            label=f"{nb} bins",
            zorder=3,
        )

    ax.axhline(
        1,
        linestyle="--",
        color="black",
        linewidth=0.8,
        alpha=0.75,
        zorder=1,
    )

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    if ylim is not None:
        ax.set_ylim(*ylim)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="0.90", linewidth=0.7, zorder=0)

    ax.legend(frameon=False, loc="best")

    fig.tight_layout(pad=0.5)

    if output_pdf is not None:
        fig.savefig(output_pdf)
    if output_png is not None:
        fig.savefig(output_png, dpi=300)

    plt.show()
    return fig


# def plot_RD_two_definitions_by_nbins(
#     rd_df,
#     ycols=("RD_strict_prob", "RD_broad_prob"),
#     titles=("Strict ratio", "Broad ratio"),
#     xlabel=None,
#     x_plot_col=None,
#     plot_bin_numbers=None,
#     output_pdf=None,
#     output_png=None,
#     ylim=None,
#     shade_ci=True,
# ):
#     paper_style_default_font()

#     if xlabel is None:
#         xlabel = xlabel_from_xcol(rd_df["x_col"].iloc[0])

#     if x_plot_col is None:
#         x_plot_col = RD_X_PLOT_COL if "RD_X_PLOT_COL" in globals() else "x_median"

#     df = _select_plot_bins(rd_df, plot_bin_numbers)
#     df = df.copy().sort_values(["n_bins", x_plot_col])
#     nbin_values = sorted(df["n_bins"].unique())

#     color_map, marker_map = _dynamic_style_for_bins(nbin_values)

#     fig, axes = plt.subplots(
#         1, 2,
#         figsize=(7.35, 2.95),
#         sharey=True,
#     )

#     for ax, ycol, title in zip(axes, ycols, titles):
#         for nb in nbin_values:
#             sub = df[df["n_bins"] == nb].copy().sort_values(x_plot_col)
#             color = color_map[nb]

#             if shade_ci:
#                 _add_bootstrap_shadow(ax, sub, x_plot_col, ycol, color=color)

#             ax.plot(
#                 sub[x_plot_col],
#                 sub[ycol],
#                 marker=marker_map[nb],
#                 markersize=4.2,
#                 markerfacecolor=color,
#                 markeredgecolor="black",
#                 markeredgewidth=0.35,
#                 color=color,
#                 linewidth=1.05,
#                 label=f"{nb} bins",
#                 zorder=3,
#             )

#         ax.axhline(
#             1,
#             linestyle="--",
#             color="black",
#             linewidth=0.8,
#             alpha=0.75,
#             zorder=1,
#         )

#         ax.set_title(title, pad=7)
#         ax.set_xlabel(xlabel)

#         if ylim is not None:
#             ax.set_ylim(*ylim)

#         ax.spines["top"].set_visible(False)
#         ax.spines["right"].set_visible(False)
#         ax.grid(axis="y", color="0.90", linewidth=0.7, zorder=0)

#     axes[0].set_ylabel(r"$R_D$")

#     handles, labels = axes[1].get_legend_handles_labels()
#     fig.legend(
#         handles,
#         labels,
#         loc="upper center",
#         ncol=len(nbin_values),
#         frameon=False,
#         bbox_to_anchor=(0.5, 1.05),
#     )

#     fig.subplots_adjust(
#         left=0.08,
#         right=0.99,
#         bottom=0.20,
#         top=0.78,
#         wspace=0.18,
#     )

#     if output_pdf is not None:
#         fig.savefig(output_pdf)
#     if output_png is not None:
#         fig.savefig(output_png, dpi=300)

#     plt.show()
#     return fig


RD_XLABEL = xlabel_from_xcol(RD_X_COL)

# fig_rd_selected_two = plot_RD_two_definitions_by_nbins(
#     RD_selected_bins,
#     ycols=("RD_strict_prob", "RD_broad_prob"),
#     titles=("Strict ratio", "Broad ratio"),
#     xlabel=RD_XLABEL,
#     x_plot_col=RD_X_PLOT_COL,
#     plot_bin_numbers=RD_PLOT_BIN_NUMBERS,
#     output_pdf=FIG_DIR / f"RD_{RD_OUTPUT_TAG}_two_definitions_by_selected_nbins.pdf",
#     output_png=FIG_DIR / f"RD_{RD_OUTPUT_TAG}_two_definitions_by_selected_nbins.png",
#     ylim=(0, 3.2),
#     shade_ci=True,
# )

fig_rd_selected_broad = plot_single_RD_column_by_nbins(
    RD_selected_bins,
    ycol="RD_broad_prob",
    ylabel=r"$R_D^{\mathrm{broad}}$",
    xlabel=RD_XLABEL,
    x_plot_col=RD_X_PLOT_COL,
    plot_bin_numbers=(5,10),
    output_pdf=FIG_DIR / f"RD_{RD_OUTPUT_TAG}_T_corr_broad_by_bins.pdf",
    output_png=FIG_DIR / f"RD_{RD_OUTPUT_TAG}_T_corrbroad_by_bins.png",
    ylim=(0, 2.5),
    shade_ci=True,
)


RD_selected_bins

RD_selected_bins.to_csv(
    "markov_results/T_CORR_RESULSTS_BINS.csv",
    index=False
)



In [ ]:
RD_X_COL = "T_env_end"

def safe_div(a, b):
    if b == 0 or pd.isna(b):
        return np.nan
    return a / b


def compute_RD_from_transition_matrix(cm, cm_prob):
    """
    Computes two R_D definitions from a transition count matrix
    and its row-normalised probability matrix.
    """
    C = "comfortable"
    N = "neutral"
    U = "uncomfortable"

    k_CU = cm.loc[C, U]
    k_UC = cm.loc[U, C]

    k_CNU = cm.loc[C, U] + cm.loc[N, U]
    k_UCN = cm.loc[U, C] + cm.loc[U, N]

    RD_strict_counts = safe_div(k_CU, k_UC)
    RD_broad_counts = safe_div(k_CNU, k_UCN)

    p_CU = cm_prob.loc[C, U]
    p_UC = cm_prob.loc[U, C]

    n_C = cm.loc[C].sum()
    n_N = cm.loc[N].sum()
    n_U = cm.loc[U].sum()

    p_CNU = safe_div(k_CNU, n_C + n_N)
    p_UCN = safe_div(k_UCN, n_U)

    RD_strict_prob = safe_div(p_CU, p_UC)
    RD_broad_prob = safe_div(p_CNU, p_UCN)

    return {
        "k_CU": k_CU,
        "k_UC": k_UC,
        "k_CNU": k_CNU,
        "k_UCN": k_UCN,

        "n_C_risk": n_C,
        "n_N_risk": n_N,
        "n_U_risk": n_U,
        "n_CN_risk": n_C + n_N,

        "p_CU": p_CU,
        "p_UC": p_UC,
        "p_CNU": p_CNU,
        "p_UCN": p_UCN,

        "RD_strict_counts": RD_strict_counts,
        "RD_broad_counts": RD_broad_counts,
        "RD_strict_prob": RD_strict_prob,
        "RD_broad_prob": RD_broad_prob,
    }


RD_METRICS = [
    "RD_strict_counts",
    "RD_broad_counts",
    "RD_strict_prob",
    "RD_broad_prob",
]


def make_bin_edges(values, n_bins, method="quantile"):
    """
    Build fixed bin edges from the original data.

    method="quantile" produces approximately equal-count bins.
    method="equal_width" produces equally spaced bins.
    """
    x = pd.Series(values).dropna().astype(float)

    if x.empty:
        raise ValueError("Cannot build bins: x_col has no finite values.")

    if n_bins < 1:
        raise ValueError("n_bins must be >= 1.")

    if method == "quantile":
        edges = np.quantile(x, np.linspace(0, 1, n_bins + 1))
        edges = np.unique(edges)

        if len(edges) < 2:
            raise ValueError(
                "Cannot build quantile bins because all values are identical."
            )

    elif method == "equal_width":
        xmin, xmax = x.min(), x.max()

        if np.isclose(xmin, xmax):
            raise ValueError(
                "Cannot build equal-width bins because all values are identical."
            )

        edges = np.linspace(xmin, xmax, n_bins + 1)

    else:
        raise ValueError("method must be 'quantile' or 'equal_width'.")

    # Expand extremes very slightly so the min/max values are always included.
    eps = np.finfo(float).eps * max(1.0, float(np.nanmax(np.abs(edges))))
    edges = edges.astype(float)
    edges[0] -= eps
    edges[-1] += eps

    return edges


def add_fixed_bins(df, x_col, edges, bin_col="_rd_bin_id"):
    """Assign observations to fixed numeric bins."""
    out = df.copy()
    out[bin_col] = pd.cut(
        out[x_col].astype(float),
        bins=edges,
        labels=False,
        include_lowest=True,
        duplicates="drop",
    )
    out = out.dropna(subset=[bin_col]).copy()
    out[bin_col] = out[bin_col].astype(int)
    return out


def _interval_label(left, right):
    return f"({left:.3g}, {right:.3g}]"


def _compute_RD_rows_from_binned(
    binned,
    x_col,
    n_bins_requested,
    edges,
    binning,
    state_order=STATE_ORDER,
    bin_col="_rd_bin_id",
):
    """Compute base R_D rows and transition matrices for already-binned data."""
    rows = []
    tmats = {}
    actual_n_bins = len(edges) - 1

    for bin_id in range(actual_n_bins):
        sub = binned[binned[bin_col] == bin_id].copy()

        cm, cm_prob = compute_transition_matrix(sub, state_order=state_order)
        rd_vals = compute_RD_from_transition_matrix(cm, cm_prob)
        tmats[bin_id] = {"counts": cm, "prob": cm_prob}

        left = edges[bin_id]
        right = edges[bin_id + 1]

        if len(sub) > 0:
            x_min_obs = sub[x_col].min()
            x_max_obs = sub[x_col].max()
            x_median = sub[x_col].median()
        else:
            x_min_obs = np.nan
            x_max_obs = np.nan
            x_median = 0.5 * (left + right)

        row = {
            "x_col": x_col,
            "binning": binning,
            "n_bins": n_bins_requested,
            "n_bins_actual": actual_n_bins,
            "bin_id": bin_id,
            "T_bin": _interval_label(left, right),
            "x_edge_left": left,
            "x_edge_right": right,
            "x_min": x_min_obs,
            "x_max": x_max_obs,
            "x_mid": 0.5 * (left + right),
            "x_median": x_median,
            "n_transitions": len(sub),
        }

        row.update(rd_vals)
        rows.append(row)

    return pd.DataFrame(rows), tmats


def _bootstrap_RD_for_binned(
    binned,
    n_bins_requested,
    actual_n_bins,
    n_boot=1000,
    cluster_cols=("ID", "walk_id"),
    seed=123,
    ci=(2.5, 97.5),
    state_order=STATE_ORDER,
    bin_col="_rd_bin_id",
):
    """
    Cluster bootstrap for R_D.

    The bin edges are fixed before resampling. Each bootstrap replicate resamples
    complete clusters, where a cluster is usually one participant in one walk.
    """
    if n_boot is None or n_boot <= 0:
        return pd.DataFrame({
            "n_bins": [n_bins_requested] * actual_n_bins,
            "bin_id": list(range(actual_n_bins)),
        })

    available_cluster_cols = [c for c in cluster_cols if c in binned.columns]

    if len(available_cluster_cols) == 0:
        # Fallback: row bootstrap if cluster columns are missing.
        groups = [binned.iloc[[i]].copy() for i in range(len(binned))]
    else:
        groups = [
            g.copy()
            for _, g in binned.groupby(available_cluster_cols, sort=False, dropna=False)
        ]

    if len(groups) == 0:
        return pd.DataFrame({
            "n_bins": [n_bins_requested] * actual_n_bins,
            "bin_id": list(range(actual_n_bins)),
        })

    rng = np.random.default_rng(seed)
    boot_rows = []

    for boot_id in range(n_boot):
        sampled_idx = rng.integers(0, len(groups), size=len(groups))
        boot = pd.concat([groups[i] for i in sampled_idx], ignore_index=True)

        for bin_id in range(actual_n_bins):
            sub = boot[boot[bin_col] == bin_id].copy()
            cm, cm_prob = compute_transition_matrix(sub, state_order=state_order)
            rd_vals = compute_RD_from_transition_matrix(cm, cm_prob)

            row = {
                "boot_id": boot_id,
                "n_bins": n_bins_requested,
                "bin_id": bin_id,
            }
            for metric in RD_METRICS:
                row[metric] = rd_vals[metric]

            boot_rows.append(row)

    boot_df = pd.DataFrame(boot_rows)
    summary_rows = []

    q_low, q_high = ci

    for bin_id in range(actual_n_bins):
        row = {
            "n_bins": n_bins_requested,
            "bin_id": bin_id,
        }

        sub_bin = boot_df[boot_df["bin_id"] == bin_id]

        for metric in RD_METRICS:
            values = sub_bin[metric].replace([np.inf, -np.inf], np.nan).dropna()

            row[f"{metric}_boot_valid"] = len(values)

            if len(values) == 0:
                row[f"{metric}_low"] = np.nan
                row[f"{metric}_high"] = np.nan
                row[f"{metric}_boot_median"] = np.nan
            else:
                row[f"{metric}_low"] = np.nanpercentile(values, q_low)
                row[f"{metric}_high"] = np.nanpercentile(values, q_high)
                row[f"{metric}_boot_median"] = np.nanmedian(values)

        summary_rows.append(row)

    return pd.DataFrame(summary_rows)


def compute_RD_bins_like_transition_matrices(
    transitions,
    x_col,
    bin_numbers=(12, 10, 8, 6, 5),
    binning="quantile",
    bootstrap=True,
    n_boot=1000,
    cluster_cols=("ID", "walk_id"),
    ci=(2.5, 97.5),
    seed=123,
    state_order=STATE_ORDER,
    output_csv=None,
):
    """
    Compute R_D in configurable bins.

    Important choices:
    - binning="quantile": approximately the same number of transitions per bin.
    - binning="equal_width": equal-width intervals along x_col.
    - bootstrap=True: adds percentile confidence bands by cluster bootstrap.
    """
    if x_col not in transitions.columns:
        raise ValueError(
            f"x_col='{x_col}' is not in transitions. "
            f"Available temperature columns include: "
            f"{[c for c in transitions.columns if c.startswith('T_')]}"
        )

    all_rows = []
    all_tmats = {}

    base = transitions.dropna(subset=[x_col, "state", "next_state"]).copy()

    for n_bins in bin_numbers:
        edges = make_bin_edges(base[x_col], n_bins=n_bins, method=binning)
        actual_n_bins = len(edges) - 1

        if actual_n_bins != n_bins:
            print(
                f"Requested {n_bins} bins for {x_col}, but only "
                f"{actual_n_bins} unique bins could be built because of tied values."
            )

        binned = add_fixed_bins(base, x_col=x_col, edges=edges)
        rd_base, tmats = _compute_RD_rows_from_binned(
            binned=binned,
            x_col=x_col,
            n_bins_requested=n_bins,
            edges=edges,
            binning=binning,
            state_order=state_order,
        )

        if bootstrap:
            rd_ci = _bootstrap_RD_for_binned(
                binned=binned,
                n_bins_requested=n_bins,
                actual_n_bins=actual_n_bins,
                n_boot=n_boot,
                cluster_cols=cluster_cols,
                seed=seed + int(n_bins),
                ci=ci,
                state_order=state_order,
            )
            rd_base = rd_base.merge(rd_ci, on=["n_bins", "bin_id"], how="left")

        all_rows.append(rd_base)
        all_tmats[n_bins] = tmats

    out = pd.concat(all_rows, ignore_index=True)

    if output_csv is not None:
        out.to_csv(output_csv, index=False)

    return out, all_tmats


def xlabel_from_xcol(x_col):
    label_map = {
        "T_corr_start": r"$T_{\mathrm{corr},i}$",
        "T_corr_end": r"$T_{\mathrm{corr},i+1}$",
        "T_corr_mean": r"$\overline{T}_{\mathrm{corr}}$",
        "T_env_start": r"$T_{\mathrm{env},i}$",
        "T_env_end": r"$T_{\mathrm{env},i+1}$",
        "T_env_mean": r"$\overline{T}_{\mathrm{env}}$",
    }
    return label_map.get(x_col, x_col)


RD_OUTPUT_TAG = f"{RD_X_COL}_{RD_BINNING}"

RD_selected_bins_Tenv, tmats_RD_selected_bins_Tenv = compute_RD_bins_like_transition_matrices(
    transitions,
    x_col=RD_X_COL,
    bin_numbers=RD_BIN_NUMBERS,
    binning=RD_BINNING,
    bootstrap=RD_RUN_BOOTSTRAP,
    n_boot=RD_BOOTSTRAP_N,
    cluster_cols=RD_BOOTSTRAP_CLUSTER_COLS,
    ci=RD_BOOTSTRAP_CI,
    seed=RD_RANDOM_SEED,
    output_csv=CSV_DIR / f"RD_bins_{RD_OUTPUT_TAG}.csv",
)

print("R_D coordinate:", RD_X_COL)
print("Binning:", RD_BINNING)
print("Computed bin numbers:", RD_BIN_NUMBERS)
print("Plotted bin numbers:", RD_PLOT_BIN_NUMBERS)
print("Bootstrap:", RD_RUN_BOOTSTRAP, "| n_boot:", RD_BOOTSTRAP_N)

display_cols = [
    "n_bins", "bin_id", "T_bin", "x_median", "n_transitions",
    "n_C_risk", "n_U_risk",
    "RD_strict_prob", "RD_strict_prob_low", "RD_strict_prob_high",
    "RD_broad_prob", "RD_broad_prob_low", "RD_broad_prob_high",
]

available_display_cols = [c for c in display_cols if c in RD_selected_bins_Tenv.columns]
display(RD_selected_bins_Tenv[available_display_cols].head(30))

## 9. Plot \(R_D\) by selected bin numbers

The plot functions below read `RD_PLOT_BIN_NUMBERS`. Change that tuple in the configuration cell to decide which bin counts are shown, without recomputing all of the notebook logic. The bootstrap interval is drawn as a shaded band around each curve when the `_low` and `_high` columns exist.


In [ ]:
def _select_plot_bins(rd_df, plot_bin_numbers=None):
    df = rd_df.copy()

    if plot_bin_numbers is not None:
        plot_bin_numbers = tuple(plot_bin_numbers)
        df = df[df["n_bins"].isin(plot_bin_numbers)].copy()

    if df.empty:
        raise ValueError(
            "No rows left to plot. Check RD_PLOT_BIN_NUMBERS against "
            "the n_bins values present in rd_df."
        )

    return df


def _dynamic_style_for_bins(nbin_values):
    nbin_values = list(nbin_values)
    cmap_values = np.linspace(0.35, 0.90, max(len(nbin_values), 2))
    colors = {
        nb: plt.cm.Blues(cmap_values[i])
        for i, nb in enumerate(nbin_values)
    }

    marker_cycle = ["o", "s", "^", "D", "v", "P", "X", "*", "<", ">"]
    markers = {
        nb: marker_cycle[i % len(marker_cycle)]
        for i, nb in enumerate(nbin_values)
    }

    return colors, markers


def _add_bootstrap_shadow(
    ax,
    sub,
    xcol,
    ycol,
    color,
    alpha=0.18,
):
    low_col = f"{ycol}_low"
    high_col = f"{ycol}_high"

    if low_col not in sub.columns or high_col not in sub.columns:
        return

    x = sub[xcol].astype(float).to_numpy()
    low = sub[low_col].astype(float).to_numpy()
    high = sub[high_col].astype(float).to_numpy()

    mask = np.isfinite(x) & np.isfinite(low) & np.isfinite(high)

    if mask.sum() >= 2:
        ax.fill_between(
            x[mask],
            low[mask],
            high[mask],
            color=color,
            alpha=alpha,
            linewidth=0,
            zorder=2,
        )
    elif mask.sum() == 1:
        # With one bin there is no area to fill, so draw a vertical interval.
        ax.vlines(
            x[mask],
            low[mask],
            high[mask],
            color=color,
            alpha=0.45,
            linewidth=1.0,
            zorder=2,
        )


def plot_single_RD_column_by_nbins(
    rd_df,
    ycol="RD_broad_prob",
    ylabel=r"$R_D$",
    xlabel=None,
    x_plot_col=None,
    plot_bin_numbers=None,
    output_pdf=None,
    output_png=None,
    ylim=None,
    shade_ci=True,
):
    paper_style_default_font()

    if xlabel is None:
        xlabel = xlabel_from_xcol(rd_df["x_col"].iloc[0])

    if x_plot_col is None:
        x_plot_col = RD_X_PLOT_COL if "RD_X_PLOT_COL" in globals() else "x_median"

    df = _select_plot_bins(rd_df, plot_bin_numbers)
    df = df.copy().sort_values(["n_bins", x_plot_col])
    nbin_values = sorted(df["n_bins"].unique())

    color_map, marker_map = _dynamic_style_for_bins(nbin_values)

    fig, ax = plt.subplots(figsize=(3.65, 2.75))

    for nb in nbin_values:
        sub = df[df["n_bins"] == nb].copy().sort_values(x_plot_col)
        color = color_map[nb]

        if shade_ci:
            _add_bootstrap_shadow(ax, sub, x_plot_col, ycol, color=color)

        ax.plot(
            sub[x_plot_col],
            sub[ycol],
            marker=marker_map[nb],
            markersize=4.2,
            markerfacecolor=color,
            markeredgecolor="black",
            markeredgewidth=0.35,
            color=color,
            linewidth=1.05,
            label=f"{nb} bins",
            zorder=3,
        )

    ax.axhline(
        1,
        linestyle="--",
        color="black",
        linewidth=0.8,
        alpha=0.75,
        zorder=1,
    )

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    if ylim is not None:
        ax.set_ylim(*ylim)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="0.90", linewidth=0.7, zorder=0)

    ax.legend(frameon=False, loc="best")

    fig.tight_layout(pad=0.5)

    if output_pdf is not None:
        fig.savefig(output_pdf)
    if output_png is not None:
        fig.savefig(output_png, dpi=300)

    plt.show()
    return fig


# def plot_RD_two_definitions_by_nbins(
#     rd_df,
#     ycols=("RD_strict_prob", "RD_broad_prob"),
#     titles=("Strict ratio", "Broad ratio"),
#     xlabel=None,
#     x_plot_col=None,
#     plot_bin_numbers=None,
#     output_pdf=None,
#     output_png=None,
#     ylim=None,
#     shade_ci=True,
# ):
#     paper_style_default_font()

#     if xlabel is None:
#         xlabel = xlabel_from_xcol(rd_df["x_col"].iloc[0])

#     if x_plot_col is None:
#         x_plot_col = RD_X_PLOT_COL if "RD_X_PLOT_COL" in globals() else "x_median"

#     df = _select_plot_bins(rd_df, plot_bin_numbers)
#     df = df.copy().sort_values(["n_bins", x_plot_col])
#     nbin_values = sorted(df["n_bins"].unique())

#     color_map, marker_map = _dynamic_style_for_bins(nbin_values)

#     fig, axes = plt.subplots(
#         1, 2,
#         figsize=(7.35, 2.95),
#         sharey=True,
#     )

#     for ax, ycol, title in zip(axes, ycols, titles):
#         for nb in nbin_values:
#             sub = df[df["n_bins"] == nb].copy().sort_values(x_plot_col)
#             color = color_map[nb]

#             if shade_ci:
#                 _add_bootstrap_shadow(ax, sub, x_plot_col, ycol, color=color)

#             ax.plot(
#                 sub[x_plot_col],
#                 sub[ycol],
#                 marker=marker_map[nb],
#                 markersize=4.2,
#                 markerfacecolor=color,
#                 markeredgecolor="black",
#                 markeredgewidth=0.35,
#                 color=color,
#                 linewidth=1.05,
#                 label=f"{nb} bins",
#                 zorder=3,
#             )

#         ax.axhline(
#             1,
#             linestyle="--",
#             color="black",
#             linewidth=0.8,
#             alpha=0.75,
#             zorder=1,
#         )

#         ax.set_title(title, pad=7)
#         ax.set_xlabel(xlabel)

#         if ylim is not None:
#             ax.set_ylim(*ylim)

#         ax.spines["top"].set_visible(False)
#         ax.spines["right"].set_visible(False)
#         ax.grid(axis="y", color="0.90", linewidth=0.7, zorder=0)

#     axes[0].set_ylabel(r"$R_D$")

#     handles, labels = axes[1].get_legend_handles_labels()
#     fig.legend(
#         handles,
#         labels,
#         loc="upper center",
#         ncol=len(nbin_values),
#         frameon=False,
#         bbox_to_anchor=(0.5, 1.05),
#     )

#     fig.subplots_adjust(
#         left=0.08,
#         right=0.99,
#         bottom=0.20,
#         top=0.78,
#         wspace=0.18,
#     )

#     if output_pdf is not None:
#         fig.savefig(output_pdf)
#     if output_png is not None:
#         fig.savefig(output_png, dpi=300)

#     plt.show()
#     return fig



RD_selected_bins_Tenv = pd.read_csv("markov_results/T_ENV_RESULSTS_BINS.csv")

RD_XLABEL = xlabel_from_xcol(RD_X_COL)

# fig_rd_selected_two_Tenv = plot_RD_two_definitions_by_nbins(
#     RD_selected_bins_Tenv,
#     ycols=("RD_strict_prob", "RD_broad_prob"),
#     titles=("Strict ratio", "Broad ratio"),
#     xlabel=RD_XLABEL,
#     x_plot_col=RD_X_PLOT_COL,
#     plot_bin_numbers=RD_PLOT_BIN_NUMBERS,
#     output_pdf=FIG_DIR / f"RD_{RD_OUTPUT_TAG}_two_definitions_by_selected_nbins.pdf",
#     output_png=FIG_DIR / f"RD_{RD_OUTPUT_TAG}_two_definitions_by_selected_nbins.png",
#     ylim=(0, 3.2),
#     shade_ci=True,
# )

fig_rd_selected_broad_Tenv = plot_single_RD_column_by_nbins(
    RD_selected_bins_Tenv,
    ycol="RD_broad_prob",
    ylabel=r"$R_D^{\mathrm{broad}}$",
    xlabel=RD_XLABEL,
    x_plot_col=RD_X_PLOT_COL,
    plot_bin_numbers=  (5,10),
    output_pdf=FIG_DIR / f"RD_{RD_OUTPUT_TAG}_broad_by_bins.pdf",
    output_png=FIG_DIR / f"RD_{RD_OUTPUT_TAG}_broad_by_bins.png",
    ylim=(0, 2.5),
    shade_ci=True,
)

RD_selected_bins_Tenv.to_csv(
    "markov_results/T_ENV_RESULSTS_BINS.csv",
    index=False
)

## 10. Optional compact moving-threshold \(R_D\) figure

Use this section only if you have dataframes like `rd_T_t1_above` with columns:

- `threshold`
- `R_D`
- `R_D_low`
- `R_D_high`

In [ ]:
def plot_RD_Tcorr_next(
    rd,
    output_pdf=FIG_DIR / "RD_Tcorr_next_stop.pdf",
    output_png=FIG_DIR / "RD_Tcorr_next_stop.png",
    t_low=26.10,
    t_high=30.77,
    ylim=(0, 3),
):
    paper_style_default_font()

    rd = rd.copy().sort_values("threshold")

    fig, ax = plt.subplots(figsize=(3.25, 2.35))

    xmin = rd["threshold"].min()
    xmax = rd["threshold"].max()

    ax.axvspan(xmin, t_low, color="#deebf7", alpha=0.55, lw=0, zorder=0)
    ax.axvspan(t_low, t_high, color="#f7fbff", alpha=0.85, lw=0, zorder=0)
    ax.axvspan(t_high, xmax, color="#bdd7e7", alpha=0.45, lw=0, zorder=0)

    for x in [t_low, t_high]:
        ax.axvline(
            x,
            linestyle="--",
            color="0.45",
            linewidth=0.8,
            alpha=0.7,
            zorder=1,
        )

    ax.plot(
        rd["threshold"],
        rd["R_D"],
        marker="o",
        markersize=3.6,
        markerfacecolor=BLUE_DARK,
        markeredgecolor="black",
        markeredgewidth=0.3,
        color=BLUE_DARK,
        linewidth=1.0,
        label=r"$R_D$",
        zorder=3,
    )

    ax.fill_between(
        rd["threshold"],
        rd["R_D_low"],
        rd["R_D_high"],
        color=BLUE_LIGHT,
        alpha=0.22,
        linewidth=0,
        zorder=2,
    )

    ax.axhline(
        1,
        linestyle="--",
        color="black",
        linewidth=0.8,
        alpha=0.7,
        label=r"$R_D=1$",
        zorder=1,
    )

    ax.set_xlabel(r"$T_{\mathrm{corr},i+1}$ threshold ($^\circ$C)")
    ax.set_ylabel(r"$R_D$")
    ax.set_ylim(*ylim)
    ax.set_xlim(xmin, xmax)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="0.90", linewidth=0.7, zorder=0)

    ax.legend(
        frameon=False,
        loc="upper left",
        handlelength=1.6,
        borderpad=0.2,
        labelspacing=0.25,
    )

    fig.tight_layout(pad=0.4)

    if output_pdf is not None:
        fig.savefig(output_pdf)
    if output_png is not None:
        fig.savefig(output_png, dpi=300)

    plt.show()
    return fig


# Example usage:
# fig = plot_RD_Tcorr_next(rd_T_t1_above)

## 11. Suggested LaTeX captions

```latex
\caption{\textbf{Transition matrices by arrival thermal regime.}
Rows indicate the comfort state at stop $i$, columns the state at stop $i+1$.
Transitions are grouped by the arrival temperature $T_{i+1}$, where the new vote
is reported. The upper row uses $T_{\mathrm{corr}}$ and the lower row uses
$T_{\mathrm{env}}$. In both coordinates, the dominant persistent state changes
from comfort in the cool regime to discomfort in the hot regime, showing the
transition-level counterpart of the static comfort--discomfort crossover.}
\label{fig:transition_matrices_temp}
```

```latex
\caption{\textbf{Same-regime persistence check.}
Persistence probabilities for transitions that start and end within the same
thermal regime. Points show $C\to C$ and $U\to U$, with Wilson 95\% confidence
intervals. The diagonal flip survives this restriction: comfort is the
persistent state in the cool regime, while discomfort becomes the persistent
state in the hot regime.}
\label{fig:same_regime_persistence}
```

```latex
\caption{\textbf{Thermal route geometry between consecutive stops.}
Rows indicate the thermal regime at the departure stop $i$, and columns the
thermal regime at the arrival stop $i+1$. Values show
$P(T_{i+1}\ \mathrm{bin}\mid T_i\ \mathrm{bin})$, with counts in parentheses.
Hot departures are not necessarily followed by hot arrivals, showing that
conditioning on $T_i$ and $T_{i+1}$ selects different subsets of transitions.}
\label{fig:route_geometry}
```